# Substation Shifting Proportionality Test

**Question:** Is the amount of load shifting per substation (Tempo − Control, TOU − Control) proportional to the number of households in that substation?

**Tests used:**
- **Pearson correlation** — tests linear association between n_households and total shifting
- **OLS regression with intercept** — verifies whether intercept ≈ 0 (necessary condition for proportionality)
- **OLS regression through origin** — directly models `shifting = k × n_households`; `k` is the per-household shift rate

**Why not ANOVA?** ANOVA compares group means across categories. Substation is not a treatment group — it's a continuous predictor. ANOVA can't express a proportional relationship.

**Why not chi-square?** Chi-square requires categorical data. Both our variables are continuous (kWh, household counts). Binning them would be arbitrary and lossy.

**TOU hour restriction:** The TOU tariff only applies during hours 13–24 (1pm–midnight); hours 1–12 are identical to control. TOU shifting is therefore computed as `sum(tou_h13-24) − sum(ctrl_h13-24)` — both sides restricted to the same tariff window so the comparison is apples-to-apples. Tempo uses all 24 hours as before.

n = 21 substations (one observation per substation)

In [1]:
import os
import zipfile
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

warnings.filterwarnings('ignore')

BASE_DIR    = '/home/user/capstone_visuals'
ZIP_PATH    = os.path.join(BASE_DIR, 'dist_net.zip')
CONTROL_CSV = os.path.join(BASE_DIR, 'data', 'control_profile.csv')
TEMPO_CSV   = os.path.join(BASE_DIR, 'data', 'tempo_shifted_profile.csv')
TOU_ZIP     = os.path.join(BASE_DIR, 'data', 'tou_shifted_profile.zip')
EXTRACT_DIR = '/tmp/dist_net'
OUTPUT_DIR  = os.path.join(BASE_DIR, 'output')

HOUR_COLS     = [str(i) for i in range(1, 25)]   # all hours — used for Tempo
TOU_HOUR_COLS = [str(i) for i in range(13, 25)]  # TOU tariff window: hours 13-24 only

def normalize_hid(x):
    try:    return str(int(float(x)))
    except: return None

print('Imports OK')
print(f'Tempo hours:  {HOUR_COLS[0]}–{HOUR_COLS[-1]}  ({len(HOUR_COLS)} hours)')
print(f'TOU hours:    {TOU_HOUR_COLS[0]}–{TOU_HOUR_COLS[-1]}  ({len(TOU_HOUR_COLS)} hours)')

Imports OK
Tempo hours:  1–24  (24 hours)
TOU hours:    13–24  (12 hours)


## 1. Build HID → Substation Mapping from Shapefiles

In [2]:
# Extract network zip if needed
flag = os.path.join(EXTRACT_DIR, '.done')
if not os.path.exists(flag):
    print('Extracting dist_net.zip...')
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(EXTRACT_DIR)
    open(flag, 'w').close()

# Load all H-nodes across substations to build HID → substation map
content = os.path.join(EXTRACT_DIR, 'content', 'output')
all_nodes = []

for folder in sorted(os.listdir(content)):
    fp = os.path.join(content, folder, f'{folder}-nodelist-HID.shp')
    if not os.path.exists(fp):
        continue
    gdf = gpd.read_file(fp)
    gdf['substation'] = folder
    all_nodes.append(gdf)

nodes = pd.concat(all_nodes, ignore_index=True)
h_nodes = nodes[nodes['label'] == 'H'].copy()
h_nodes['hid_key'] = h_nodes['hid'].apply(normalize_hid)
h_nodes = h_nodes.dropna(subset=['hid_key'])

# Household count per substation
hh_counts = h_nodes.groupby('substation')['hid_key'].nunique().rename('n_households')

print(f'Total H-nodes: {len(h_nodes):,}')
print(f'Substations:   {hh_counts.shape[0]}')
print()
print(hh_counts.sort_values(ascending=False).to_string())

Total H-nodes: 28,221
Substations:   21

substation
139356    4536
121578    3719
121579    3146
121581    2428
121582    1780
121575    1277
147948    1184
121577    1098
156947     994
121587     978
121586     906
147966     906
147965     904
147947     822
121580     807
144353     729
121583     511
121584     455
139355     370
156257     370
147963     297


## 2. Load CSVs and Compute Per-Substation Total Shifting

**Shifting metric choice:**
- **Tempo:** net shift = `sum(tempo_h1-24) − sum(ctrl_h1-24)`. Tempo moves load out of (or into) periods, so the signed sum meaningfully reflects net load change.
- **TOU:** TOU redistributes load *within* hours 13–24 (some hours decrease, others increase), so the signed net across the full window cancels to ~0 and is uninformative. Instead we use `Σ|tou_h − ctrl_h|` for h in 13–24 — the **sum of per-hour absolute differences** — which captures total load movement regardless of direction.

In [3]:
# Load control
print('Loading control...')
ctrl = pd.read_csv(CONTROL_CSV)
ctrl['hid_key'] = ctrl['hid'].apply(normalize_hid)

# Load tempo
print('Loading tempo...')
tempo = pd.read_csv(TEMPO_CSV)
tempo['hid_key'] = tempo['hid'].apply(normalize_hid)

# Load TOU (from zip)
print('Loading TOU...')
with zipfile.ZipFile(TOU_ZIP) as z:
    with z.open('tou_shifted_profile.csv') as f:
        tou = pd.read_csv(f)
tou['hid_key'] = tou['hid'].apply(normalize_hid)

print(f'Control rows:  {len(ctrl):,}   unique HIDs: {ctrl["hid_key"].nunique():,}')
print(f'Tempo rows:    {len(tempo):,}   unique HIDs: {tempo["hid_key"].nunique():,}')
print(f'TOU rows:      {len(tou):,}   unique HIDs: {tou["hid_key"].nunique():,}')

Loading control...


Loading tempo...


Loading TOU...


Control rows:  197,365   unique HIDs: 28,195
Tempo rows:    197,365   unique HIDs: 28,195
TOU rows:      197,365   unique HIDs: 28,195


In [4]:
# Merge HID → substation into each dataset
hid_sub = h_nodes[['hid_key', 'substation']].drop_duplicates(subset='hid_key')

def add_substation(df):
    return df.merge(hid_sub, on='hid_key', how='inner')

ctrl_s  = add_substation(ctrl)
tempo_s = add_substation(tempo)
tou_s   = add_substation(tou)

print(f'After merge — Control: {len(ctrl_s):,} rows | Tempo: {len(tempo_s):,} rows | TOU: {len(tou_s):,} rows')

After merge — Control: 197,365 rows | Tempo: 197,365 rows | TOU: 197,365 rows


In [5]:
# Compute per-substation shifting.
#
# Tempo: net shift over all 24h = sum(tempo) - sum(ctrl)
# TOU:   total hourly movement over tariff window (h13-24)
#        = sum of |tou_h - ctrl_h| per HID-day, aggregated to substation.
#        The signed net cancels to ~0 because TOU redistributes *within* the window.

def substation_total_load(df, hour_cols):
    df = df.copy()
    df['window_total'] = df[hour_cols].sum(axis=1)
    return df.groupby('substation')['window_total'].sum()

def substation_hourly_abs_diff(df_a, df_b, hour_cols):
    """
    For each matched (hid_key, date) row, compute sum(|a_h - b_h|) for h in hour_cols,
    then aggregate to substation. Requires both dataframes to have 'substation' column.
    """
    merged = df_a[['hid_key', 'date', 'substation'] + hour_cols].merge(
        df_b[['hid_key', 'date'] + hour_cols],
        on=['hid_key', 'date'], suffixes=('_a', '_b')
    )
    abs_diff_cols = []
    for h in hour_cols:
        col = f'absdiff_{h}'
        merged[col] = (merged[f'{h}_a'] - merged[f'{h}_b']).abs()
        abs_diff_cols.append(col)
    merged['row_abs_diff'] = merged[abs_diff_cols].sum(axis=1)
    return merged.groupby('substation')['row_abs_diff'].sum()

# Tempo: net shift (signed and absolute of net)
load_ctrl_full = substation_total_load(ctrl_s,  HOUR_COLS)
load_tempo     = substation_total_load(tempo_s, HOUR_COLS)

# TOU: per-hour absolute differences over tariff window
tou_abs_movement = substation_hourly_abs_diff(ctrl_s, tou_s, TOU_HOUR_COLS)

df = pd.DataFrame({
    'n_households':    hh_counts,
    'load_ctrl_full':  load_ctrl_full,
    'load_tempo':      load_tempo,
    'tou_abs_movement': tou_abs_movement,
}).dropna()

df['shift_tempo']     = df['load_tempo'] - df['load_ctrl_full']
df['abs_shift_tempo'] = df['shift_tempo'].abs()
# For TOU the single meaningful metric is total hourly movement
df['shift_tou']       = df['tou_abs_movement']   # always >= 0

print(f'Substations in analysis: {len(df)}')
print()
print(df[['n_households', 'shift_tempo', 'abs_shift_tempo', 'shift_tou']]
      .sort_values('n_households', ascending=False)
      .rename(columns={'shift_tou': 'tou_hourly_movement'})
      .to_string())

Substations in analysis: 21

            n_households  shift_tempo  abs_shift_tempo  tou_hourly_movement
substation                                                                 
139356              4536  -142.857407       142.857407            2113.7094
121578              3719    10.281413        10.281413            1789.5585
121579              3146   198.843901       198.843901            1527.9399
121581              2428   223.369565       223.369565            1093.0026
121582              1780   -94.505789        94.505789             916.0686
121575              1277   -90.794945        90.794945             691.8624
147948              1184   -59.507804        59.507804             579.5358
121577              1098   -84.099111        84.099111             581.1384
156947               994   -51.068391        51.068391             311.8923
121587               978    26.965057        26.965057             431.9586
121586               906    75.856390        75.856390     

## 3. Pearson Correlation

Tests H₀: ρ = 0 (no linear association between n_households and shifting).  
Tempo is tested for both signed and absolute shifting. TOU uses only the hourly-movement metric.

In [6]:
def pearson_report(x, y, label_x, label_y):
    r, p = stats.pearsonr(x, y)
    n = len(x)
    z  = np.arctanh(r)
    se = 1 / np.sqrt(n - 3)
    ci_lo, ci_hi = np.tanh(z - 1.96*se), np.tanh(z + 1.96*se)
    print(f'  {label_y} ~ {label_x}')
    print(f'    n = {n}')
    print(f'    r = {r:+.4f}   (95% CI: [{ci_lo:+.4f}, {ci_hi:+.4f}])')
    print(f'    p = {p:.4f}  {"***" if p<0.001 else "**" if p<0.01 else "*" if p<0.05 else "(ns)"}')
    print()
    return r, p

print('=== Pearson Correlation: n_households vs shifting ===')
print()
print('--- TEMPO ---')
pearson_report(df['n_households'], df['shift_tempo'],     'n_households', 'shift_tempo (signed)')
pearson_report(df['n_households'], df['abs_shift_tempo'], 'n_households', 'shift_tempo (absolute)')

print('--- TOU (hourly movement, hours 13-24) ---')
pearson_report(df['n_households'], df['shift_tou'], 'n_households', 'tou_hourly_movement')

=== Pearson Correlation: n_households vs shifting ===

--- TEMPO ---
  shift_tempo (signed) ~ n_households
    n = 21
    r = +0.0882   (95% CI: [-0.3571, +0.5008])
    p = 0.7037  (ns)

  shift_tempo (absolute) ~ n_households
    n = 21
    r = +0.5834   (95% CI: [+0.2027, +0.8109])
    p = 0.0055  **

--- TOU (hourly movement, hours 13-24) ---
  tou_hourly_movement ~ n_households
    n = 21
    r = +0.9939   (95% CI: [+0.9848, +0.9976])
    p = 0.0000  ***



(np.float64(0.9939484560366597), np.float64(1.0820479476337781e-19))

## 4. OLS Regression

**With intercept:** checks whether the intercept is statistically distinguishable from zero.  
**Through origin (no intercept):** the strict proportionality model `shifting = k × n_households`.  

Uses `scipy.stats.linregress` (with intercept) and a manual OLS formula for the origin-constrained version.

In [7]:
def ols_report(x, y, label_y, scenario):
    x_arr = x.values.astype(float)
    y_arr = y.values.astype(float)
    n = len(x_arr)

    # With intercept
    res = stats.linregress(x_arr, y_arr)
    print(f'  [{scenario}] {label_y} ~ n_households')
    print(f'  OLS WITH intercept:')
    print(f'    slope     = {res.slope:+.6f}  (se={res.stderr:.6f},  p={res.pvalue:.4f} {"***" if res.pvalue<0.001 else "**" if res.pvalue<0.01 else "*" if res.pvalue<0.05 else "(ns)"})')
    print(f'    intercept = {res.intercept:+.4f}  (se={res.intercept_stderr:.4f},  t={res.intercept/res.intercept_stderr:+.3f})')
    print(f'    R²        = {res.rvalue**2:.4f}')

    # Through origin: k = (x·y)/(x·x)
    k = np.dot(x_arr, y_arr) / np.dot(x_arr, x_arr)
    residuals = y_arr - k * x_arr
    ss_res = np.sum(residuals**2)
    ss_tot = np.sum((y_arr - y_arr.mean())**2)
    r2_origin = 1 - ss_res / ss_tot
    sigma2 = ss_res / (n - 1)
    se_k   = np.sqrt(sigma2 / np.dot(x_arr, x_arr))
    t_k    = k / se_k
    p_k    = 2 * stats.t.sf(abs(t_k), df=n-1)

    print(f'  OLS THROUGH ORIGIN (strict proportionality):')
    print(f'    k (kWh/household) = {k:+.6f}  (se={se_k:.6f},  t={t_k:+.3f},  p={p_k:.4f} {"***" if p_k<0.001 else "**" if p_k<0.01 else "*" if p_k<0.05 else "(ns)"})')
    print(f'    R²                = {r2_origin:.4f}')
    print()
    return res, k, se_k

print('=== OLS Regression ===')
print()
print('--- TEMPO: SIGNED ---')
res_tempo,  k_tempo,  se_tempo  = ols_report(df['n_households'], df['shift_tempo'],     'shift_tempo',     'TEMPO')
print('--- TEMPO: ABSOLUTE ---')
res_atempo, k_atempo, se_atempo = ols_report(df['n_households'], df['abs_shift_tempo'], 'abs_shift_tempo', 'TEMPO')
print('--- TOU: HOURLY MOVEMENT (h13-24) ---')
res_tou,    k_tou,    se_tou    = ols_report(df['n_households'], df['shift_tou'],        'tou_hourly_movement', 'TOU')

=== OLS Regression ===

--- TEMPO: SIGNED ---
  [TEMPO] shift_tempo ~ n_households
  OLS WITH intercept:
    slope     = +0.006935  (se=0.017964,  p=0.7037 (ns))
    intercept = -9.3185  (se=31.5302,  t=-0.296)
    R²        = 0.0078
  OLS THROUGH ORIGIN (strict proportionality):
    k (kWh/household) = +0.002871  (se=0.011291,  t=+0.254,  p=0.8019 (ns))
    R²                = 0.0032

--- TEMPO: ABSOLUTE ---
  [TEMPO] abs_shift_tempo ~ n_households
  OLS WITH intercept:
    slope     = +0.028674  (se=0.009159,  p=0.0055 **)
    intercept = +30.7508  (se=16.0755,  t=+1.913)
    R²        = 0.3403
  OLS THROUGH ORIGIN (strict proportionality):
    k (kWh/household) = +0.042086  (se=0.006272,  t=+6.710,  p=0.0000 ***)
    R²                = 0.2133

--- TOU: HOURLY MOVEMENT (h13-24) ---
  [TOU] tou_hourly_movement ~ n_households
  OLS WITH intercept:
    slope     = +0.482110  (se=0.012223,  p=0.0000 ***)
    intercept = -20.2462  (se=21.4550,  t=-0.944)
    R²        = 0.9879
  OLS THRO

## 5. Visualisation

In [8]:
BG   = '#06090f'
TXT  = '#c8d8e8'
DIM  = '#2a3f55'
C_TEMPO = '#ff8c00'
C_TOU   = '#00b4d8'
C_FIT   = '#ffffff'

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG,
    'text.color': TXT, 'axes.labelcolor': TXT,
    'xtick.color': TXT, 'ytick.color': TXT,
    'axes.edgecolor': DIM, 'grid.color': DIM,
    'font.family': 'monospace', 'font.size': 9,
})

x_range = np.linspace(0, df['n_households'].max() * 1.05, 200)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor=BG)
fig.suptitle('Substation Shifting vs. Household Count\n'
             'Solid = OLS with intercept   |   Dashed = through-origin (proportionality)',
             color='white', fontsize=11, fontweight='bold', y=1.03)

configs = [
    (axes[0], df['shift_tempo'],     C_TEMPO, 'TEMPO',  'Signed Net Shift (kWh, all 24h)',      res_tempo,  k_tempo ),
    (axes[1], df['abs_shift_tempo'], C_TEMPO, 'TEMPO',  'Absolute Net Shift (kWh, all 24h)',    res_atempo, k_atempo),
    (axes[2], df['shift_tou'],       C_TOU,   'TOU',    'Hourly Movement (kWh, h13–24)',        res_tou,    k_tou   ),
]

for ax, y_data, color, scenario, ylabel, res_obj, k_val in configs:
    x_vals = df['n_households'].values
    y_vals = y_data.values

    r, p = stats.pearsonr(x_vals, y_vals)
    sig   = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'

    ax.scatter(x_vals, y_vals, color=color, s=55, zorder=5, alpha=0.85,
               edgecolors='white', linewidths=0.4)

    for sub, xi, yi in zip(df.index, x_vals, y_vals):
        ax.annotate(sub, (xi, yi), textcoords='offset points', xytext=(5, 3),
                    fontsize=5.5, color=TXT, alpha=0.7)

    y_fit  = res_obj.slope * x_range + res_obj.intercept
    ax.plot(x_range, y_fit,       color=C_FIT, lw=1.3, alpha=0.7, zorder=4, label='OLS fit')
    ax.plot(x_range, k_val*x_range, color=C_FIT, lw=1.3, alpha=0.7, zorder=4,
            linestyle='--', label='Through origin')

    ax.axhline(0, color=DIM, lw=0.8, zorder=2)
    ax.set_xlabel('Households per substation', fontsize=8)
    ax.set_ylabel(ylabel, fontsize=8)
    ax.set_title(f'{scenario}  |  r = {r:+.3f}  (p = {p:.3f}, {sig})', fontsize=9)
    ax.grid(lw=0.35, alpha=0.5)
    ax.legend(fontsize=7, facecolor='#0c1622', edgecolor=DIM, labelcolor=TXT)

plt.tight_layout()
out_path = os.path.join(OUTPUT_DIR, '07_shifting_proportionality.png')
fig.savefig(out_path, dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print(f'Saved → {out_path}')

Saved → /home/user/capstone_visuals/output/07_shifting_proportionality.png


## 6. Summary Table

In [9]:
rows = []
for scenario, measure, y_col, res_obj, k_val, se_k_val in [
    ('Tempo', 'Signed net (24h)',        'shift_tempo',     res_tempo,  k_tempo,  se_tempo ),
    ('Tempo', 'Absolute net (24h)',      'abs_shift_tempo', res_atempo, k_atempo, se_atempo),
    ('TOU',   'Hourly movement (h13-24)','shift_tou',       res_tou,    k_tou,    se_tou   ),
]:
    x = df['n_households'].values.astype(float)
    y = df[y_col].values.astype(float)
    r, p = stats.pearsonr(x, y)
    rows.append({
        'Scenario': scenario,
        'Shifting measure': measure,
        'Pearson r': round(r, 4),
        'p-value':  round(p, 4),
        'Sig. (α=0.05)': 'Yes' if p < 0.05 else 'No',
        'OLS slope': round(res_obj.slope, 6),
        'OLS intercept': round(res_obj.intercept, 4),
        'OLS R²': round(res_obj.rvalue**2, 4),
        'k (origin, kWh/HH)': round(k_val, 6),
        'k SE': round(se_k_val, 6),
    })

summary = pd.DataFrame(rows)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
print(summary.to_string(index=False))

Scenario         Shifting measure  Pearson r  p-value Sig. (α=0.05)  OLS slope  OLS intercept  OLS R²  k (origin, kWh/HH)     k SE
   Tempo         Signed net (24h)     0.0882   0.7037            No   0.006935        -9.3185  0.0078            0.002871 0.011291
   Tempo       Absolute net (24h)     0.5834   0.0055           Yes   0.028674        30.7508  0.3403            0.042086 0.006272
     TOU Hourly movement (h13-24)     0.9939   0.0000           Yes   0.482110       -20.2462  0.9879            0.473280 0.007843
